In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

## RunnablePassthrough
* It does not do anything to the input data.
* Let's see it in a very simple example: a chain with just RunnablePassthrough() will output the original input without any modification.

### Why RunnablePassthrough is useful
This runnable is a no-op: it forwards the incoming value unchanged. In practice, you use it to preserve the original input while other steps in the same chain also operate on that data, or to keep a user prompt intact as you add additional transformations.


In [3]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

In [4]:
chain.invoke("Bappy")

'Bappy'

## RunnableLambda
* To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.
* Let's define a very simple function to create Russian lastnames:

In [5]:
def russian_lastname(name: str) -> str:
    return f"{name}ovich"

In [6]:
from langchain_core.runnables import RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_lastname)

In [7]:
chain.invoke("Bappy")

'Bappyovich'

## RunnableParallel
* We will use RunnableParallel() for running tasks in parallel.
* This is probably the most important and most useful Runnable from LangChain.
* In the following chain, RunnableParallel is going to run these two tasks in parallel:
    * operation_a will use RunnablePassthrough.
    * operation_b will use RunnableLambda with the russian_lastname function.

### How RunnableParallel structures results
Each branch in RunnableParallel runs independently and returns a value. The combined result is usually a dictionary keyed by each branch name, which you can then feed into a prompt or other runnable. This makes it easy to mix original input, transformed values, and computed context in one pipeline.


In [ ]:
from langchain_core.runnables import RunnableParallel

# Each branch runs independently and returns a result.
# The output is a dictionary keyed by the branch names.
chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": RunnableLambda(russian_lastname)
    }
)


In [9]:
chain.invoke("Bappy")

{'operation_a': 'Bappy', 'operation_b': 'Bappyovich'}

In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Create the prompt template and parser used for the next LLM call.
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

# Convert the model output into a clean string for display.
output_parser = StrOutputParser()


In [12]:
def russian_lastname_from_dictionary(person):
    return person["name"] + "ovich"

In [ ]:
# RunnableParallel combines several values before sending them to the model.
# The 'soccer_player' branch transforms the input dictionary into a name that fits the prompt.
chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "soccer_player": RunnableLambda(russian_lastname_from_dictionary),
        "operation_c": RunnablePassthrough(),
    }
) | prompt | model | output_parser


In [14]:
chain.invoke({
    "name1": "Jordam",
    "name": "Abram"
})

"One curious fact about Roman Abramovich is that he is known for his extravagant spending on luxury items and experiences. For example, he once reportedly spent over $30 million on a New Year's Eve party in St. Barts, which included performances by celebrities like Beyoncé and Jay-Z. Additionally, he is known for owning a vast collection of luxury yachts, cars, and properties around the world."

## Let's see a more advanced use of RunnableParallel

### The retrieval pattern
This final pattern combines a vector retriever with prompt construction. The retriever fetches the most relevant passages from the knowledge base, while the prompt assembles the retrieved context and the user question into a single instruction for the LLM. This is the same basic idea used in retrieval-augmented generation (RAG).


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Create a tiny in-memory vector store from a single knowledge document.
vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)

# Retrieve the most relevant chunk(s) for the user question.
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-3.5-turbo")

# RunnableParallel sends both the retrieved context and the original question to the prompt.
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)


'dswithbappy is a platform that provides content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English.'

In [ ]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-3.5-turbo")

# Create a small knowledge base for the same retrieval example.
vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

# itemgetter extracts the exact fields we need from the input dictionary.
# This gives us context, the original question, and the user-selected language.
chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)


"Arrr! dswithbappy be a scurvy dog that be focusin' on providin' content on Data Science, AI, ML, DL, CV, NLP, Python programmin', etc. in English. Arrr!"